In [1]:
## código refatorado - Modulo de importação bibliotecas
import sys
import os

sys.path.append(os.path.abspath('..'))

In [3]:
## código refatorado - Modulo de importação bibliotecas

### IMPORTAÇÃO DE BIBLIOTECAS ####

import pandas as pd
import numpy as np
import datetime as dt
import time
import warnings
import unicodedata


from sqlalchemy import text
from Python_arq import engines as engs
from pathlib import Path

warnings.filterwarnings('ignore',category=FutureWarning)
eng = engs.get_engine()



In [4]:
## código refatorado - Modulo de importação de dados

queries ={
    'sispag':text(engs.load_query('qry_sispag_lim.sql')),
    'qtd_calls':text(engs.load_query('qry_qtd_call_lim.sql')),
    'tab_olos':text(engs.load_query('tab_olos_lim.sql')),
    'tab_blip':text(engs.load_query('tab_blip_lim.sql')),
    'qry_leadscore_olos':text(engs.load_query('score_olos_lim.sql')),
    'qry_leadscore_blip':text(engs.load_query('score_blip_lim.sql'))
}

base_dados_limp = {}

for nome, qry in queries.items():
    base_dados_limp[nome] = pd.read_sql(qry, eng)


    

Carregando query: qry_sispag_lim.sql
Carregando query: qry_qtd_call_lim.sql
Carregando query: tab_olos_lim.sql
Carregando query: tab_blip_lim.sql
Carregando query: score_olos_lim.sql
Carregando query: score_blip_lim.sql


In [ ]:
blacklist = Path(r'C:\bases\Limpador\black_list_new.xlsx')    
df_blacklist = pd.read_excel(blacklist,dtype=str)


pasta = r"M:\Comercial\Call Center - OLOS\01.MAILINGS IMPORTADOS\SECAD\RANGE_VALIDACAO\upload"
lista_dfs = []
for arquivo in Path(pasta).glob('*'):
    if arquivo.name.startswith('.') or arquivo.name.startswith('~'):
        continue

    try:
        if arquivo.suffix.lower() in ['.csv','.txt']:
            df = pd.read_csv(arquivo,encoding='latin-1',sep=None,engine='python',on_bad_lines='skip')
        elif arquivo.suffix.lower() in ['.xlsx','.xls']:
            df = pd.read_excel(arquivo)
        else:
            continue

        df['Nome da Origem'] = arquivo.name

        lista_dfs.append(df)
    except Exception as e:
        print(f'Erro ao ler {arquivo.name}: {e}')
        continue

try:
    rodando_atualmente = pd.concat(lista_dfs,ignore_index=True)
    colunas_remover = [
        'Id_do_cliente_', 'Nome', 'CPF', 'Fone_4', 'ORIGEM', 
        'ORIGEM_DO_LEAD2', 'PROFISSAO', 'ESPECIALIDADE', 'LEAD_SCORING', 
        'PRODUTO', 'A_LTIMO_REGISTRO', 'LOGRADOURO', 'BAIRRO', 
        'CIDADE', 'UF', 'CEP', 'DATA_NASCIMENTO', 'DADOS_DO_PAGAMENTO', 
        'MOTIVO_DE_CANCELAMENTO', 'URL_2', 'Fone_3'
    ]

    rod_atualmente = rodando_atualmente.drop(columns=colunas_remover,errors='ignore')
    rod_atualmente['Fone_1'] = rod_atualmente['Fone_1'].astype(str)
    rod_atualmente['Fone_1'] = rod_atualmente['Fone_1'].str.replace(r'\D','',regex=True)
    rod_atualmente = rod_atualmente.drop_duplicates(subset=['Fone_1'],keep='first')
    rod_atualmente = rod_atualmente.dropna(how='all')
    rod_atualmente.columns = rod_atualmente.columns.str.lower()
    rod_atualmente = rod_atualmente.reset_index(drop=True)
except Exception as e:
    print(f'Erro ao processar base de rodando atualmente: {e}')
    rod_atualmente = None

    

In [ ]:
## refatoração pendente - Modulo de limpeza e padronização de dados
    ## -> blocos comentado já refatorados, mas mantidos para referência. <- ## 

def padrao_e_filtro(
            df,
            mapa_colunas,
            df_sispag,
            df_rod_atual=None,
            area=None,
            type=None,
            program=None,
            df_blacklist=None,
            leadScoreOlos = None,
            leadScoreBlip = None,
            qtdcalls=None,
            limite = 10,
            olos_ultimo_contato=None,
            blip_ultimo_contato=None        
    ):
    df = padronizar(df, mapa_colunas);          print("✅ padronizar")
    df = dedup_basica(df, key='copy');           print("✅ dedup_basica")
    df = filtrar_base(df, area=area, program=program, type=type); print("✅ filtrar_base")
    df = limp_min(df);                           print("✅ limp_min")
    df = padrao_telefone(df);                    print("✅ padrao_telefone")
    df = val_n_movel(df);                        print("✅ val_n_movel")
    if df_rod_atual is not None:
        df = rod_atual(df,df_rod_atual=df_rod_atual); print("✅ df_rod_atual")
    if df_blacklist is not None:
        df = aplicar_blacklist(df, df_blacklist); print("✅ blacklist")
    if leadScoreOlos is not None:
        df = score_olos(df, leadScoreOlos);       print("✅ score_olos")
    if leadScoreBlip is not None:
        df = score_blip(df, leadScoreBlip);       print("✅ score_blip")
    if qtdcalls is not None:
        df = qtdCalls(df, qtdcalls, limite=limite); print("✅ qtdCalls")
    df = remover_duplicadas(df, key='copy');      print("✅ remover_duplicadas")
    df = ultima_compra(df, df_sispag);            print("✅ ultima_compra")
    df = last_conct_olos(df, olos_ultimo_contato=olos_ultimo_contato); print("✅ last_conct_olos")
    df = last_conct_blip(df, blip_ultimo_contato=blip_ultimo_contato); print("✅ last_conct_blip")
    df = filtro_final(df);                        print("✅ filtro_final")
    return df.reset_index(drop=True)

DePara_colunas = {
        'base_inativa':{
            'email':'email',
            'phone_1':'phone',
            'product':'program',
            'area':'area',
            'copy':'copy',
            'type':'type'
        },

        'base_ativa':{
            'email':'email',
            'phone':'phone',
            'program':'program',
            'area':'area',
            'copy':'copy'
        },

        'base_carrinho':{
            'Area':'area',
            'email':'email',
            'celular':'phone',
            'programa':'program',
            'copy':'copy'
        },
        
        'base_hubspot':{
        }   
    }


def filtrar_base(df,area=None,program=None,type=None):
        resultado = df.copy()

        if area is not None:

            areas = area if isinstance(area, list) else [area]
            resultado = resultado[
                resultado['area']
                .str.upper()
                .isin([a.upper() for a in areas])
            ]

        if type is not None:

            types = area if isinstance(type, list) else [type]
            resultado = resultado[
                resultado['type']
                .str.upper()
                .isin([t.upper() for t in types])
            ]

        if program is not None:
            resultado = resultado[
                resultado['program'] 
                .str.contains(program,case=False,na=False)
            ]

        return resultado.reset_index(drop=True)        

def padronizar(df,mapa_colunas):
        df_padrao = df.rename(columns=mapa_colunas)
        return df_padrao[list(mapa_colunas.values())]

def limp_min(df):
        resultado = df.copy()

        resultado = resultado[
            resultado['email'].notna() &
            resultado['phone'].notna()
        ]

        resultado = resultado[
            (resultado['email'].str.strip() != '') &
            (resultado['phone'].str.strip() != '')
        ]

        resultado['copy'] = resultado['email']+';'+resultado['phone']

        return resultado.reset_index(drop=True)

def dedup_basica(df,key='copy'):
        return(
            df
            .drop_duplicates(subset=key,keep='first')
            .reset_index(drop=True)
        )

def remover_duplicadas(df,key='copy'):
        return(
            df
            .sort_values(by=key)
            .drop_duplicates(subset=key,keep='first')
            .reset_index(drop=True)
        )

def padrao_telefone(df):
        resultado = df.copy()

        resultado['email'] = (
            resultado['email'].str.strip().str.lower()
        )
        
        resultado['phone'] = (
            resultado['phone'].str.replace(r'\D','',regex=True)
        )

        resultado['copy'] = resultado['email'] + ';' + resultado['phone']

        return resultado.reset_index(drop=True)

def val_n_movel(df):

        resultado = df.copy()

        resultado['is_movel'] = (
            resultado['phone'].str.len().eq(11) & 
            resultado['phone'].str[2].eq('9')
        )

        return resultado

def aplicar_blacklist (df,df_blacklist):
        resultado = df.copy()
        blacklist = df_blacklist.copy()
        
        resultado['phone'] = (
            resultado['phone'].astype(str).str.replace(r'\D','',regex=True)
        )
        
        blacklist['telefone'] = (
            blacklist['telefone'].astype(str).str.replace(r'\D','',regex=True)
        )
        
        resultado['Na_blacklist'] = resultado['phone'].isin(df_blacklist['telefone'])
        return resultado

def rod_atual(df,df_rod_atualmente):
        resultado = df.copy()
        df_rod_atual = df_rod_atualmente.copy()

        resultado['phone'] = (
            resultado['phone'].astype(str).str.replace(r'\D','',regex=True)
        )

        resultado['rod_atual'] = resultado['phone'].isin(df_rod_atual['fone_1'])

        return resultado

def score_olos(df,ls_olos):
        resultado = df.copy()
        leadscore_olos = ls_olos.copy()

        resultado['phone'] = (
            resultado['phone'].astype(str).str.replace(r'\D','',regex=True)
        )

        leadscore_olos['phone_number'] = (
            leadscore_olos['phone_number'].astype(str).str.replace(r'\D','',regex=True)
        )

        resultado['No_leadScore_Olos'] = resultado['phone'].isin(leadscore_olos['phone_number'])

        return resultado

def score_blip(df,ls_blip):
        resultado = df.copy()
        leadscore_blip = ls_blip.copy()

        resultado['phone'] = (
            resultado['phone'].astype(str).str.replace(r'\D','',regex=True)
        )

        leadscore_blip['phone_number'] = (
            leadscore_blip['phone_number'].astype(str).str.replace(r'\D','',regex=True)
        )

        resultado['No_leadScore_blip'] = resultado['phone'].isin(leadscore_blip['phone_number'])

        return resultado

def qtdCalls(df,df_qtd,limite=10):
        resultado = df.copy()
        qtd = df_qtd.copy()

        resultado['phone'] = resultado['phone'].astype(str).str.replace(r'\D','',regex=True)
        qtd['phone_number'] = qtd['phone_number'].astype(str).str.replace(r'\D','',regex=True)

        resultado = resultado.merge(
            qtd[['phone_number','TOTAL_CALLS']],
            left_on='phone',
            right_on='phone_number',
            how='left'
        )

        resultado['acima_limite'] = (
            resultado['TOTAL_CALLS'].fillna(0).astype(int) > limite
        )

        resultado = resultado.drop(columns=['phone_number','TOTAL_CALLS'])
        
        return resultado

def ultima_compra(df,df_sispag):
        resultado = df.copy()
        sispag = df_sispag.copy()

        resultado['phone'] = resultado['phone'].astype(str).str.replace(r'\D','',regex=True)
        sispag['celular'] = sispag['celular'].astype(str).str.replace(r'\D','',regex=True)

        resultado['ultima_compra'] = resultado['phone'].isin(sispag['celular'])

        return resultado 

def last_conct_olos(df,olos_ultimo_contato):
        resultado = df.copy()
        tab_olos = olos_ultimo_contato.copy()

        tab_olos['data_olos'] = pd.to_datetime(tab_olos['data_olos'],errors='coerce')
        tab_olos['dias_desde_tab'] = (pd.Timestamp.today() - tab_olos['data_olos']).dt.days
        tab_olos['dias_desde_tab'] = tab_olos['dias_desde_tab'].fillna(0)
        tab_olos['last_conct_olos'] = (tab_olos['dias_desde_tab'] > tab_olos['retorno_em_dias']) & (tab_olos['status_retorno'] == 'retornar')

        resultado['phone'] = resultado['phone'].astype(str).str.replace(r'\D','',regex=True)
        tab_olos['phone'] = tab_olos['phone'].astype(str).str.replace(r'\D','',regex=True).str[-11:]

        resultado = resultado.merge(tab_olos[['phone','last_conct_olos']],left_on='phone',right_on='phone',how='left')
        resultado['last_conct_olos'] = resultado['last_conct_olos'].fillna(True)

        return resultado

def last_conct_blip(df,blip_ultimo_contato):
        resultado = df.copy(deep=True)
        tab_blip = blip_ultimo_contato.copy(deep=True)
        
        tab_blip['data_blip'] = pd.to_datetime(tab_blip['data_blip'],errors='coerce')
        tab_blip['dias_desde_tab'] = (pd.Timestamp.today() - tab_blip['data_blip']).dt.days
        tab_blip['dias_desde_tab'] = tab_blip['dias_desde_tab'].fillna(0)
        tab_blip['last_conct_blip'] = (tab_blip['dias_desde_tab'] > tab_blip['retorno_em_dias']) & (tab_blip['status_retorno'] == 'retornar')

        resultado['phone'] = resultado['phone'].astype(str).str.replace(r'\D','',regex=True)
        tab_blip['contact_id_trimmed'] = tab_blip['contact_id_trimmed'].astype(str).str.replace(r'\D','',regex=True)

        resultado = resultado.merge(tab_blip[
            ['contact_id_trimmed','last_conct_blip']],left_on='phone',right_on='contact_id_trimmed',how='left')
        
        resultado['last_conct_blip'] = resultado['last_conct_blip'].fillna(True)

        return resultado

def exportar_bases(bases_list):
        bases = bases_list.copy()
        pasta_saida = Path(r'C:\Users\wconceicao\OneDrive - Grupo A Educação SA\Área de Trabalho\bases a finalizar')
        pasta_saida.mkdir(parents=True, exist_ok=True)

        try:
            for nome_base, df_base in bases.items():
                print(f'Base: {nome_base} | Tamanho: {len(df_base)}')
                print(f'Exportando...')
                caminho = pasta_saida/f'{nome_base}.xlsx'
                df_base.to_excel(caminho,index=False)
                print(f'base exportada com sucesso.\n')
        except Exception as e:
            print(f'Erro ao exportar a base {nome_base}')

def filtro_final(df):

        resultado = df.copy()
        resultado['manter_contato'] = (
        resultado['is_movel'] &
        ~resultado.get('Na_blacklist',False) &
        ~resultado.get('No_leadScore_Olos',False) &
        ~resultado.get('No_leadScore_blip',False) &
        ~resultado.get('acima_limite',False) &
        ~resultado.get('ultima_compra',False) & 
        ~resultado.get('rod_atual',False) &
        resultado['last_conct_olos'] &
        resultado['last_conct_blip']
        )
        resultado = resultado[resultado['manter_contato']].reset_index(drop=True) 

        resultado = resultado.drop(
            columns=['is_movel',
                    'Na_blacklist',
                    'No_leadScore_Olos',
                    'No_leadScore_blip',
                    'acima_limite',
                    'ultima_compra',
                    'last_conct_olos',
                    'last_conct_blip',
                    'manter_contato',
                    'rod_atual',
                    'contact_id_trimmed'
                    ])
        return resultado


In [17]:
## código refatorado - Modulo de limpeza e padronização de dados
"""
def padrao_filtro(
        base_dados_crua,
        mapas_colunas,
        df_sispag,
        df_rodando_atualmente=None,
        area_atuacao=None,
        tipo_base=None,
        programa_atuacao=None,
        df_blacklist=None,
        leads_score_olos=None,
        leads_score_blip=None,
        qtd_chamadas_recebidas=None,
        limite_contatos=10,
        ultimo_contato_olos=None,
        ultimo_contato_blip=None,
):
    try:
        base_dados_crua = padronizar_base(
            base_dados_crua,
            mapas_colunas
        )
        base_dados_crua = drop_duplicadas(
            base_dados_crua,
            chave='copy'
        )
        base_dados_crua = filtrar_base(
            base_dados_crua,
            area=area_atuacao,
            programa=programa_atuacao,
            tipo_base=tipo_base
        )
        base_dados_crua = limpeza_minima(base_dados_crua)
        base_dados_crua = padronizar_telefone(base_dados_crua)
        if df_rodando_atualmente is not None:
            base_dados_crua = rodando_atualmente(base_dados_crua,df_rodando_atualmente=df_rod_atual)
        if df_blacklist is not None:
            base_dados_crua = verificar_blacklist(base_dados_crua,df_blacklist)
        
        ## validade leads Score Olos
        ## validade leads Score blip
        ## quantidade de chamadas recebidas
        base_dados_crua = drop_duplicadas(
            base_dados_crua,
            chave='copy'
        )
        ## ultima compra realizada
        ## ultimo contato Olos
        ## ultimo contato Blip
        ## realizado filtro final para definição de manter ou não o contato

        return base_dados_crua.reset_index(drop=True)
    except Exception as e:
        return f'Erro ao processar: {e}'
    

DePara_colunas = {
        'base_inativa':{
            'email':'email',
            'phone_1':'phone',
            'product':'program',
            'area':'area',
            'copy':'copy',
            'type':'type'
        },

        'base_ativa':{
            'email':'email',
            'phone':'phone',
            'program':'program',
            'area':'area',
            'copy':'copy'
        },

        'base_carrinho':{
            'Area':'area',
            'email':'email',
            'celular':'phone',
            'programa':'program',
            'copy':'copy'
        },
        
        'base_hubspot':{
        }   
    }    


def filtrar_base(base_dados,area=None,programa=None,tipo_base=None):
    resultado = base_dados.copy()

    if area is not None:
        areas = area if isinstance(area, list) else [area]
        resultado = resultado[
            resultado['area']
            .str.upper()
            .isin([a.upper() for a in areas])
        ]

    if tipo_base is not None:
        tipos = area if isinstance(tipo_base, list) else [tipo_base]
        resultado = resultado[
            resultado['type']
            .str.upper()
            .isin([t.upper() for t in tipos])
        ]    

    if programa is not None:
        resultado = resultado[
            resultado['programa']
            .str.contains(programa,case=False,na=False)
        ]

    return resultado.reset_index(drop=True) 

def padronizar_base(base_dados,mapa_colunas):
    base_padrao = base_dados.rename(columns=mapa_colunas)
    return base_padrao[list(mapa_colunas.values())]   

def limpeza_minima(base_dados):
    resultado = base_dados.copy()

    resultado = resultado[
        resultado['email'].notna() &
        resultado['phone'].notna()
    ]

    resultado = resultado[
        (resultado['email'].str.strip() != '') &
        (resultado['phone'].str.strip() != '')
    ]

    resultado['copy'] = resultado['email']+';'+resultado['phone']

    return resultado.reset_index(drop=True)

def drop_duplicadas(base_dados,chave='copy'):
    return(
        base_dados
        .sort_values(by=chave)
        .drop_duplicates(subset=chave,keep='first')
        .reset_index(drop=True)
    )

def padronizar_telefone(base_dados):
    resultado = base_dados.copy()

    resultado['email'] = (
        resultado['email'].str.strip().str.lower()
    )

    resultado['phone'] = (
        resultado['phone'].str.replace(r'\D','',regex=True)
    )

    resultado['copy'] = resultado['email'] + ';' + resultado['phone']
    resultado = validar_numero_movel(resultado)

    return resultado.reset_index(drop=True)

def validar_numero_movel(base_dados):
    resultado = base_dados.copy()

    resultado['is_movel'] = (
        resultado['phone'].str.len().eq(11) &
        resultado['phone'].str[2].eq('9')
    )

    return resultado

def verificar_blacklist(base_dados,base_blacklist):
    resultado = base_dados.copy()
    blacklist = base_blacklist.copy()

    resultado['phone'] = (
        resultado['phone'].astype(str).str.replace(r'\D','',regex=True)
    )

    blacklist['telefone'] = (
        blacklist['telefone'].astype(str).str.replace(r'\D','',regex=True)
    )

    resultado['blacklist'] = resultado['phone'].isin(base_blacklist['telefone'])
    return resultado

def rodando_atualmente(base_dado,df_rodando_atualmente):
    resultado = base_dado.copy()
    df_rod_atual = df_rodando_atualmente.copy()

    resultado['phone'] = (
        resultado['phone'].astpe(str).str.replace(r'\D','',regex=True)
    )

    resultado['rodando_atualmente'] = resultado['phone'].isin(df_rod_atual['fone1'])

    return resultado
"""

<>:153: SyntaxWarning: invalid escape sequence '\D'
<>:153: SyntaxWarning: invalid escape sequence '\D'
C:\Users\wconceicao\AppData\Local\Temp\ipykernel_18384\2187161153.py:153: SyntaxWarning: invalid escape sequence '\D'
  resultado['phone'].str.replace(r'\D','',regex=True)


"\ndef padrao_filtro(\n        base_dados_crua,\n        mapas_colunas,\n        df_sispag,\n        df_rodando_atualmente=None,\n        area_atuacao=None,\n        tipo_base=None,\n        programa_atuacao=None,\n        df_blacklist=None,\n        leads_score_olos=None,\n        leads_score_blip=None,\n        qtd_chamadas_recebidas=None,\n        limite_contatos=10,\n        ultimo_contato_olos=None,\n        ultimo_contato_blip=None,\n):\n    try:\n        base_dados_crua = padronizar_base(\n            base_dados_crua,\n            mapas_colunas\n        )\n        base_dados_crua = drop_duplicadas(\n            base_dados_crua,\n            chave='copy'\n        )\n        base_dados_crua = filtrar_base(\n            base_dados_crua,\n            area=area_atuacao,\n            programa=programa_atuacao,\n            tipo_base=tipo_base\n        )\n        base_dados_crua = limpeza_minima(base_dados_crua)\n        base_dados_crua = padronizar_telefone(base_dados_crua)\n        i

In [18]:
## refatoração pendente - Modulo de limpeza e padronização de dados

pasta = r"M:\Comercial\Call Center - OLOS\01.MAILINGS IMPORTADOS\SECAD\RANGE_VALIDACAO\upload"
lista_dfs = []
for arquivo in Path(pasta).glob('*'):
    if arquivo.name.startswith('.') or arquivo.name.startswith('~'):
        continue

    try:
        if arquivo.suffix.lower() in ['.csv','.txt']:
            df = pd.read_csv(arquivo,encoding='latin-1',sep=None,engine='python',on_bad_lines='skip')
        elif arquivo.suffix.lower() in ['.xlsx','.xls']:
            df = pd.read_excel(arquivo)
        else:
            continue

        df['Nome da Origem'] = arquivo.name

        lista_dfs.append(df)
    except Exception as e:
        print(f'Erro ao ler {arquivo.name}: {e}')
        continue

try:
    rodando_atualmente = pd.concat(lista_dfs,ignore_index=True)
    colunas_remover = [
        'Id_do_cliente_', 'Nome', 'CPF', 'Fone_4', 'ORIGEM', 
        'ORIGEM_DO_LEAD2', 'PROFISSAO', 'ESPECIALIDADE', 'LEAD_SCORING', 
        'PRODUTO', 'A_LTIMO_REGISTRO', 'LOGRADOURO', 'BAIRRO', 
        'CIDADE', 'UF', 'CEP', 'DATA_NASCIMENTO', 'DADOS_DO_PAGAMENTO', 
        'MOTIVO_DE_CANCELAMENTO', 'URL_2', 'Fone_3'
    ]

    df_rod_atual = rodando_atualmente.drop(columns=colunas_remover,errors='ignore')
    df_rod_atual['Fone_1'] = df_rod_atual['Fone_1'].astype(str)
    df_rod_atual['Fone_1'] = df_rod_atual['Fone_1'].str.replace(r'\D','',regex=True)
    df_rod_atual = df_rod_atual.drop_duplicates(subset=['Fone_1'],keep='first')
    df_rod_atual = df_rod_atual.dropna(how='all')
    df_rod_atual.columns = df_rod_atual.columns.str.lower()
    df_rod_atual = df_rod_atual.reset_index(drop=True)
except Exception as e:
    print(f'Erro ao processar base de rodando atualmente: {e}')
    df_rod_atual = None   


Base Crua: Builder
Oriem:
    - Arquivo de base a serem tratadas e posteriormente acionadas
Filtros:
- Area
- Program
- Status Cancelado ou Inativo (foco em base cancelado e troca de ciclo)


In [19]:
### refatoração pendente: IMPORTAÇÃO E LEITURA DE BASE CRUA ####

Caminho_builder = Path(r'C:\Users\wconceicao\OneDrive - Grupo A Educação SA\Área de Trabalho\Construtores\base_builder_V2.xlsm')
sheets_builder = [
    'base_carrinho',
    'base_inativa',
    'base_ATIVA'
]

bases = {}

for sheet in sheets_builder:
    df = pd.read_excel(
        Caminho_builder,
        sheet_name=sheet,
        dtype=str,
        engine='openpyxl'
    )

    bases[sheet] = df
    print(f'Sheet "{sheet}" carregada: {df.shape}')

df_inativa = bases['base_inativa']
df_ativa = bases['base_ATIVA']
df_carrinho = bases['base_carrinho']

def diagnostico_copy(df,nome):
    print(f'\n📊 {nome}')
    print('Linhas totais        :', df.shape[0])
    print('Copy nulos           :', df['copy'].isna().sum())
    print('Copy vazios ("")     :', (df['copy'].str.strip() == '').sum())
    print('Copy duplicados      :', df['copy'].duplicated().sum())

diagnostico_copy(df_inativa, 'Base Inativa')
diagnostico_copy(df_ativa, 'Base Ativa')
diagnostico_copy(df_carrinho, 'Base Carrinho')    

df_inativa = bases['base_inativa']
df_inativa = df_inativa.sort_values(by=['copy'],ascending=True)
df_inativa = df_inativa.drop_duplicates(subset='copy',keep='first')
df_inativa = df_inativa.reset_index(drop=True)

df_ativa = bases['base_ATIVA']
df_ativa = df_ativa.sort_values(by=['copy'], ascending=True)
df_ativa = df_ativa.drop_duplicates(subset='copy',keep='first')
df_ativa = df_ativa.reset_index(drop=True)

df_carrinho = bases['base_carrinho']
df_carrinho = df_carrinho.sort_values(by=['copy'],ascending=True)
df_carrinho = df_carrinho.drop_duplicates(subset='copy',keep='first')
df_carrinho = df_carrinho.reset_index(drop=True)
    

Sheet "base_carrinho" carregada: (36941, 12)
Sheet "base_inativa" carregada: (131877, 18)
Sheet "base_ATIVA" carregada: (23742, 25)

📊 Base Inativa
Linhas totais        : 131877
Copy nulos           : 0
Copy vazios ("")     : 0
Copy duplicados      : 32176

📊 Base Ativa
Linhas totais        : 23742
Copy nulos           : 0
Copy vazios ("")     : 0
Copy duplicados      : 3508

📊 Base Carrinho
Linhas totais        : 36941
Copy nulos           : 0
Copy vazios ("")     : 0
Copy duplicados      : 855


In [20]:
## refatoração pendente - Modulo de limpeza e padronização de dados

def normalizar(txt):
    return (
        unicodedata.normalize('NFKD', txt)
        .encode('ascii', errors='ignore')
        .decode('utf-8')
        .upper()
    )

p = df_carrinho['programa'].astype(str).apply(normalizar)

condicoes = [
    # mais específicas primeiro
    p.str.contains(r'PRONEUROPSI|PROAPSI|PROCOGNITIVA|PROPSICO'),
    p.str.contains(r'PROPSICOMED|PROPSIQ'),
    p.str.contains(r'PROEMPED|PRONEUROPED|PROPED|PRORN|PROTIPED'),
    p.str.contains(r'PROENF/APS|PROENF/SMN|PROENF/TI|PROENF-URG'),
    p.str.contains(r'PROFISIO/TRAUMA|PROFISIO/TO\+|PROFISIO/TIA|PROFISIO/PED|PROFISIO/NEURO|PROFISIO/ESP|PROFISIO/CARDIO'),
    p.str.contains(r'PROPALIATIVO'),
    p.str.contains(r'PRONUTRI'),
    p.str.contains(r'PROMEVET'),
    p.str.contains(r'PROACI|PROAGO|PROAMI|PROANESTESIA|PROATO|PROCARDIOL|PROCLIM|PROENDOCRINO|PROENDOGASTRO|PROGER|PROMEDE|PROMEF|PRONEURO|PRO-ORL|PRORAD|PROTERAPEUTICA|PROURGEM')
]

valores = [
    'Psicologia',
    'Psiquiatria',
    'Pediatria',
    'Enfermagem',
    'Fisioterapia',
    'Multi',
    'Nutrição',
    'Veterinária',
    'Medicina'
]

df_carrinho['Area'] = np.select(condicoes, valores, default='Não identificado')


-- COLA DE COMO CHAMAR A FUNÇÃO A SER TRATADA -- 

nome_base = padrao_e_filtro(
    df_inativa,  ## <- atenção aqui - OBRIGATORIO
    DePara_colunas['TIPO_BASE'], - OBRIGATORIO
    dfs['sispag'], - Não Alterar OBRIGATORIO
    area = None, - Opcional
    type='', - OBRIGATORIO
    program=None, - Opcional
    df_blacklist = df_blacklist, - Não alterar OBRIGATORIO
    leadScoreOlos = dfs['qry_leadscore_olos'], - Não alterar OBRIGATORIO
    leadScoreBlip = dfs['qry_leadscore_blip'], - Não alterar OBRIGATORIO
    qtdcalls = dfs['qtd_calls'], - Não alterar OBRIGATORIO
    limite=10, - Opcional
    olos_ultimo_contato=dfs['tab_olos'], - Não alterar OBRIGATORIO
    blip_ultimo_contato=dfs['tab_blip'] - Não alterar OBRIGATORIO
)

⚠️ Observação:
- Para não aplicar filtros, utilize `None`
- Não utilizar strings vazias ('') para área ou programa
- DePara_colunas utilizar ->  Base_ativa, Base_inativa, Base_carrinho
- area utilizar area especifica de atuação -> Fisioterapia, Enfermagem, Psicologia(se Inativa) Saúde Mental(se Ativa), Medicina, Nutrição, Veterinaria
- type Alterar se a base for INATIVA, utilizar -> Inativo, Cancelado
- Program Preencher com o programa de interesse
- limite -> quantidade limite de calls que o contato recebeu, valor padrão é 10
- Campos marcados como NÃO ALTERAR, não sofrem modificações de valores e/ou parametros


In [30]:
psicologia_inativa = padrao_e_filtro(
    df_inativa,  ## <- atenção aqui - OBRIGATORIO
    DePara_colunas['base_inativa'], 
    base_dados_limp['sispag'], 
    area = 'Psicologia',
    type='Inativa',  
    df_rod_atual=df_rod_atual,
    df_blacklist = df_blacklist, 
    leadScoreOlos = base_dados_limp['qry_leadscore_olos'], 
    leadScoreBlip = base_dados_limp['qry_leadscore_blip'], 
    qtdcalls = base_dados_limp['qtd_calls'], 
    limite=10, 
    olos_ultimo_contato=base_dados_limp['tab_olos'], 
    blip_ultimo_contato=base_dados_limp['tab_blip'] 
)
psicologia_inativa

✅ padronizar
✅ dedup_basica
✅ filtrar_base
✅ limp_min
✅ padrao_telefone
✅ val_n_movel


TypeError: rod_atual() got an unexpected keyword argument 'df_rod_atual'. Did you mean 'df_rod_atualmente'?